# ETL and Data Cleaning

## Objectives

This notebook will:

- extract the raw online retail transaction data;
- inspect the dataset structure and data types;
- assess missing values and duplicated records;
- identify cancellations, returns, and invalid values;
- document data-quality limitations;
- transform the data without changing the original CSV;
- save a cleaned dataset for later analysis.

The original file stored in `data/raw` will remain unchanged.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## Extract the raw dataset

The project uses `pathlib` to construct a reliable path to the raw dataset. The code works when the notebook is opened from either the project root or the `jupyter_notebooks` directory.

In [2]:
project_root = Path.cwd()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "online_retail.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

print(f"Project root: {project_root}")
print(f"Dataset: {data_path}")

Project root: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project2-Online Retail Transaction Analysis/CI-DA-Project-2-Online-Retail-Transaction-Analysis
Dataset: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project2-Online Retail Transaction Analysis/CI-DA-Project-2-Online-Retail-Transaction-Analysis/data/raw/online_retail.csv


In [3]:
df_raw = pd.read_csv(
    data_path,
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "CustomerID": "Int64",
    },
    parse_dates=["InvoiceDate"],
)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

Rows: 541,909
Columns: 8


In [4]:
df_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## Customer identifier quality investigation

Customer identifiers are required for customer-level analysis and RFM
segmentation. Before using them, we need to check for missing values, unusual
frequencies, and identifiers associated with implausibly diverse activity.

In [5]:
customer_id_quality = pd.Series(
    {
        "Total rows": len(df_raw),
        "Missing CustomerID values": df_raw["CustomerID"].isna().sum(),
        "Unique CustomerID values": df_raw["CustomerID"].nunique(),
    }
)

customer_id_quality

Total rows                   541909
Missing CustomerID values         0
Unique CustomerID values       4372
dtype: int64

In [7]:
customer_frequency = (
    df_raw["CustomerID"]
    .value_counts()
    .rename_axis("CustomerID")
    .reset_index(name="RowCount")
)

customer_frequency.head(10)

,CustomerID,RowCount
0,15287,135101
1,17841,7983
2,14911,5903
3,14096,5128
4,12748,4642
5,14606,2782
6,15311,2491
7,14646,2085
8,13089,1857
9,13263,1677


In [8]:
most_frequent_count = customer_frequency.loc[0, "RowCount"]
second_most_frequent_count = customer_frequency.loc[1, "RowCount"]

frequency_comparison = pd.Series(
    {
        "Most frequent CustomerID": customer_frequency.loc[0, "CustomerID"],
        "Most frequent row count": most_frequent_count,
        "Second-highest row count": second_most_frequent_count,
        "Ratio to second-highest": (
            most_frequent_count / second_most_frequent_count
        ),
        "Percentage of all rows": (
            most_frequent_count / len(df_raw) * 100
        ),
    }
)

frequency_comparison

Most frequent CustomerID    15,287.00
Most frequent row count    135,101.00
Second-highest row count     7,983.00
Ratio to second-highest         16.92
Percentage of all rows          24.93
dtype: float64

In [9]:
top_customers = customer_frequency.head(10).copy()
top_customers["CustomerID"] = top_customers["CustomerID"].astype(str)

fig = px.bar(
    top_customers,
    x="CustomerID",
    y="RowCount",
    title="Top 10 Customer Identifiers by Transaction Row Count",
    labels={
        "CustomerID": "Customer identifier",
        "RowCount": "Number of transaction rows",
    },
    color="RowCount",
    color_continuous_scale="Blues",
    text_auto=",",
)

fig.update_layout(
    xaxis_type="category",
    coloraxis_showscale=False,
)

fig.show()

In [10]:
suspected_customer_id = 15287

customer_15287 = df_raw.loc[
    df_raw["CustomerID"].eq(suspected_customer_id)
].copy()

customer_15287_summary = pd.Series(
    {
        "Transaction rows": len(customer_15287),
        "Percentage of all rows": (
            len(customer_15287) / len(df_raw) * 100
        ),
        "Unique invoices": customer_15287["InvoiceNo"].nunique(),
        "Unique products": customer_15287["StockCode"].nunique(),
        "Unique countries": customer_15287["Country"].nunique(),
        "First transaction": customer_15287["InvoiceDate"].min(),
        "Last transaction": customer_15287["InvoiceDate"].max(),
    }
)

customer_15287_summary

Transaction rows                       135101
Percentage of all rows                  24.93
Unique invoices                          3713
Unique products                          3811
Unique countries                            9
First transaction         2010-12-01 11:52:00
Last transaction          2011-12-09 10:26:00
dtype: object

In [11]:
customer_15287_countries = (
    customer_15287.groupby("Country", as_index=False)
    .size()
    .rename(columns={"size": "RowCount"})
    .sort_values("RowCount", ascending=False)
)

customer_15287_countries

,Country,RowCount
7,United Kingdom,133621
1,EIRE,711
3,Hong Kong,288
8,Unspecified,202
6,Switzerland,125
2,France,66
4,Israel,47
5,Portugal,39
0,Bahrain,2


In [12]:
fig = px.bar(
    customer_15287_countries,
    x="Country",
    y="RowCount",
    title="Country Distribution for Customer Identifier 15287",
    labels={
        "Country": "Country",
        "RowCount": "Number of transaction rows",
    },
    text_auto=",",
)

fig.update_layout(xaxis={"categoryorder": "total descending"})

fig.show()

### Possible median imputation

`CustomerID` is an identifier, not a numerical measurement, so its median has no customer-behaviour meaning. However, calculating it can help investigate whether a preprocessing step replaced missing identifiers with the median identifier value.

In [13]:
customer_id_median = df_raw["CustomerID"].median()

print(f"Median CustomerID: {customer_id_median:,.0f}")
print(
    "Does the most frequent identifier equal the median?",
    customer_id_median == suspected_customer_id,
)

Median CustomerID: 15,287
Does the most frequent identifier equal the median? True


## Raw-data integrity conclusion

The raw DataFrame is stored as `df_raw`. All cleaning and feature engineering will be performed on a separate copy so that the original imported data remains available for comparison.

The customer identifier investigation found that `15287` occurs in 135,101 rows, representing approximately 24.93% of the dataset. It is nearly 17 times more frequent than the next customer identifier and is associated with 3,713 invoices, 3,811 products, nine countries, and activity spanning almost the entire dataset period.

The value `15287` is also the median customer identifier. Because customer identifiers are categorical labels rather than meaningful numerical
measurements, this combination of findings strongly suggests that the supplied CSV may have replaced unknown customer identifiers with the median identifier.

This is an inference because the supplied CSV does not include a preprocessing record that confirms how the value was created.

The affected transaction rows will remain available for overall sales, product, time, and geographic analysis because their transaction information remains useful. However, identifier `15287` will be excluded from customer-level RFM segmentation because treating all of these rows as one customer would produce a misleading customer profile.

This approach may also exclude some genuine transactions belonging to the real customer `15287`. That trade-off will be documented as a project limitation.